In [ ]:
# ============================================
# DFD03 Celeb-DF v2 — FULL PREPROCESSING
# ============================================
# Input: "celeb-df-v2" dataset from Kaggle
# Output: preprocessed_CelebDF_16.tar.gz + preprocessed_CelebDF_32.tar.gz
#
# Celeb-DF v2 has folders: Celeb-real/, Celeb-synthesis/, YouTube-real/
# Our preprocess_videos.py expects: real/ and fake/
# So we create symlinks to map the structure.
#
# IMPORTANT: Pillow must be downgraded BEFORE other imports
# because Kaggle's pre-installed Pillow 11+ breaks torchvision.

NUM_FRAMES_LIST = [16, 32]  # Process both T=16 and T=32
OUTPUT_SIZE = 224
MIN_FACE_CONF = 0.90
MIN_DETECTION_RATIO = 0.55
DETECTOR_MAX_SIDE = 960

import subprocess, sys, os, time, shutil, json
import glob as glob_mod

# Step 1: Force downgrade Pillow FIRST
print('=== Downgrading Pillow ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'],
               check=True)
print('Pillow downgraded.')

# Step 2: Install other dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch',
                'opencv-python-headless', 'tqdm'],
               check=True)
print('Dependencies installed.')

# Step 3: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 4: Explore Celeb-DF v2 dataset structure
print('\n=== Exploring Celeb-DF v2 dataset structure ===')
celeb_input = None
for candidate in [
    '/kaggle/input/celeb-df-v2',
    '/kaggle/input/datasets/reubensuju/celeb-df-v2',
]:
    if os.path.exists(candidate):
        celeb_input = candidate
        print(f'Found dataset at: {celeb_input}')
        break

if celeb_input is None:
    print('Searching for dataset...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'celeb' in root.lower():
            celeb_input = root
            print(f'Found: {celeb_input}')
            break

if celeb_input is None:
    raise RuntimeError('Celeb-DF v2 dataset not found!')

# Show structure
print('\nDataset structure:')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}
for root, dirs, files in os.walk(celeb_input):
    level = root.replace(celeb_input, '').count(os.sep)
    if level < 3:
        vids = sum(1 for f in files if os.path.splitext(f)[1].lower() in VIDEO_EXTS)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {vids} videos, {len(files)} files)')

# Step 5: Create symlinked structure real/ + fake/
# Celeb-DF v2 has: Celeb-real, Celeb-synthesis, YouTube-real
# Our preprocess_videos.py needs folders named: real/ and fake/
# \"synthesis\" is not in our keyword list, so we create symlinks.
PREPARED_DIR = '/kaggle/working/celeb_prepared'
if os.path.exists(PREPARED_DIR):
    shutil.rmtree(PREPARED_DIR)
os.makedirs(os.path.join(PREPARED_DIR, 'real'), exist_ok=True)
os.makedirs(os.path.join(PREPARED_DIR, 'fake'), exist_ok=True)

real_count = 0
fake_count = 0

for root, dirs, files in os.walk(celeb_input):
    dirname = os.path.basename(root).lower()
    
    for f in files:
        if os.path.splitext(f)[1].lower() not in VIDEO_EXTS:
            continue
        
        src = os.path.join(root, f)
        
        # Determine label from folder name
        if 'real' in dirname or 'youtube' in dirname or 'original' in dirname:
            dst = os.path.join(PREPARED_DIR, 'real', f)
            real_count += 1
        elif 'synthesis' in dirname or 'fake' in dirname or 'swap' in dirname or 'manipulated' in dirname:
            dst = os.path.join(PREPARED_DIR, 'fake', f)
            fake_count += 1
        else:
            print(f'  SKIP unknown folder: {root}/{f}')
            continue
        
        # Avoid filename duplicates
        if os.path.exists(dst):
            name, ext = os.path.splitext(f)
            dst = os.path.join(os.path.dirname(dst), f'{name}_{dirname}{ext}')
        
        os.symlink(src, dst)

print(f'\nPrepared: {real_count} real, {fake_count} fake symlinks')

# Step 6: Process for each T value
for NUM_FRAMES in NUM_FRAMES_LIST:
    output_name = f'preprocessed_CelebDF_{NUM_FRAMES}'
    output_dir = f'/kaggle/working/{output_name}'
    
    print(f'\n{"="*60}')
    print(f'PREPROCESSING Celeb-DF v2 T={NUM_FRAMES}')
    print(f'  Input: {PREPARED_DIR}')
    print(f'  Output: {output_dir}')
    print(f'{"="*60}\n')
    
    cmd = [
        sys.executable, '-u',
        '/kaggle/working/project/preprocess_videos.py',
        PREPARED_DIR,
        output_dir,
        '--max-frames', str(NUM_FRAMES),
        '--output-size', str(OUTPUT_SIZE),
        '--min-face-confidence', str(MIN_FACE_CONF),
        '--min-detection-ratio', str(MIN_DETECTION_RATIO),
        '--detector-max-side', str(DETECTOR_MAX_SIDE),
        '--device', 'cuda',
    ]
    
    print(f'Command: {" ".join(cmd)}')
    
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        line = line.rstrip('\n')
        if line:
            print(line, flush=True)
    proc.wait()
    elapsed = (time.time() - t0) / 60
    
    if proc.returncode != 0:
        print(f'\nERROR: preprocessing T={NUM_FRAMES} failed (code {proc.returncode})')
        print('Continuing to next T value...')
        continue
    
    print(f'\nT={NUM_FRAMES} done in {elapsed:.1f} min')
    
    # Count results
    real_out = len([d for d in os.listdir(os.path.join(output_dir, 'real'))
                    if os.path.isdir(os.path.join(output_dir, 'real', d))]) if os.path.isdir(os.path.join(output_dir, 'real')) else 0
    fake_out = len([d for d in os.listdir(os.path.join(output_dir, 'fake'))
                    if os.path.isdir(os.path.join(output_dir, 'fake', d))]) if os.path.isdir(os.path.join(output_dir, 'fake')) else 0
    print(f'  Real: {real_out}, Fake: {fake_out}, Total: {real_out + fake_out}')
    
    # Package
    tar_name = f'{output_name}.tar.gz'
    tar_path = f'/kaggle/working/{tar_name}'
    print(f'\nPackaging {tar_name}...')
    os.system(f'tar czf {tar_path} -C /kaggle/working {output_name}/')
    tar_size = os.path.getsize(tar_path) / (1024**2)
    print(f'{tar_name}: {tar_size:.1f} MB')

# Final summary
print(f'\n{"="*60}')
print('CELEB-DF V2 PREPROCESSING COMPLETE')
print(f'{"="*60}')
for NUM_FRAMES in NUM_FRAMES_LIST:
    tar_name = f'preprocessed_CelebDF_{NUM_FRAMES}.tar.gz'
    tar_path = f'/kaggle/working/{tar_name}'
    if os.path.exists(tar_path):
        sz = os.path.getsize(tar_path) / (1024**2)
        print(f'  {tar_name}: {sz:.1f} MB')
print('Upload these as Kaggle datasets for multi-dataset training.')